In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = os.path.expanduser("~/Desktop/europrocure-intelligence/data/raw/export_CAN_2023_2018.csv")

# Load sample for missingness analysis
df = pd.read_csv(
    DATA_PATH,
    nrows=50000,
    encoding='utf-8',
    low_memory=False
)

print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")

Loaded 50,000 rows and 75 columns


In [3]:
# Calculate missingness for all 75 columns
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('missing_pct', ascending=False)

# Split into three groups
high_missing = missing[missing['missing_pct'] >= 70]
medium_missing = missing[(missing['missing_pct'] > 0) & (missing['missing_pct'] < 70)]
complete = missing[missing['missing_pct'] == 0]

print(f"HIGH MISSING (>=70%) — DROP CANDIDATES: {len(high_missing)} columns")
print(high_missing.to_string())

print(f"\nMEDIUM MISSING (1-69%) — KEEP WITH CARE: {len(medium_missing)} columns")
print(medium_missing.to_string())

print(f"\nCOMPLETE (0% missing) — BACKBONE COLUMNS: {len(complete)} columns")
print(complete.to_string())

HIGH MISSING (>=70%) — DROP CANDIDATES: 8 columns
                         missing_count  missing_pct
ISO_COUNTRY_CODE_ALL             49993        99.99
EU_INST_CODE                     49952        99.90
B_ACCELERATED                    48879        97.76
INFO_ON_NON_AWARD                44077        88.15
FRA_ESTIMATED                    39724        79.45
NUMBER_TENDERS_NON_EU            36783        73.57
NUMBER_TENDERS_OTHER_EU          36616        73.23
WIN_NATIONALID                   36359        72.72

MEDIUM MISSING (1-69%) — KEEP WITH CARE: 47 columns
                              missing_count  missing_pct
NUMBER_TENDERS_SME                    33046        66.09
CRIT_PRICE_WEIGHT                     32515        65.03
AWARD_EST_VALUE_EURO                  31271        62.54
NUMBER_OFFERS_ELECTR                  31246        62.49
CRIT_WEIGHTS                          27509        55.02
CRIT_CRITERIA                         27475        54.95
CAE_NATIONALID                

In [4]:
# Visualise missingness as a horizontal bar chart
fig = px.bar(
    missing[missing['missing_pct'] > 0].reset_index(),
    x='missing_pct',
    y='index',
    orientation='h',
    color='missing_pct',
    color_continuous_scale=['#2ecc71', '#f39c12', '#e74c3c'],
    title='Missingness by Column — TED Contract Award Notices 2018-2023',
    labels={'missing_pct': 'Missing (%)', 'index': 'Column'},
    height=900
)

fig.add_vline(
    x=70,
    line_dash='dash',
    line_color='red',
    annotation_text='70% threshold — drop above this line',
    annotation_position='top right'
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    plot_bgcolor='white',
    coloraxis_showscale=False
)

fig.show()

## Missingness Visualisation — Key Decisions

The chart above shows missingness across all 75 columns sorted from highest to lowest.

The red dashed line marks the 70% threshold. Any column above this line is dropped from the analysis.

**8 columns confirmed for removal:**
- ISO_COUNTRY_CODE_ALL, EU_INST_CODE, B_ACCELERATED, INFO_ON_NON_AWARD
- FRA_ESTIMATED, NUMBER_TENDERS_NON_EU, NUMBER_TENDERS_OTHER_EU, WIN_NATIONALID

**Backbone columns (green — 0% missing):**
- ID_NOTICE_CAN, TOP_TYPE, YEAR, TYPE_OF_CONTRACT, ISO_COUNTRY_CODE
- VALUE_EURO_FIN_2, CPV, DT_DISPATCH, DT_AWARD

These backbone columns form the foundation of every mart table and business query.